In [2]:
import subprocess
import sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'qiskit==2.2.3','qiskit-machine-learning==0.9.0','numpy==2.2.6',
    'scipy==1.15.3','scikit-learn==1.6.1','pandas==2.2.3',
    'matplotlib==3.10.1','flask==3.1.0'])

0

In [12]:
import numpy as np
answers=np.array([3,5,2,4],float)
assert np.all((answers>=1)&(answers<=5))
normalized=(answers-1)/4
print('성향 점수:',normalized.tolist())

규칙 점수: 1.5
교육용 그룹: 1


In [14]:
import time
from pathlib import Path
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from qiskit.circuit.library import z_feature_map, zz_feature_map, pauli_feature_map
from qiskit.circuit.library import real_amplitudes, efficient_su2, n_local, pauli_two_design
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.primitives import QMLSampler, QMLEstimator
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import VQC

def parity(integer):
    return int(integer).bit_count() % 2

def circuits(n, fmap_name='Z', ansatz_name='RealAmplitudes', reps=1):
    if fmap_name == 'Z':
        fmap = z_feature_map(n, reps=1)
    elif fmap_name == 'ZZ':
        fmap = zz_feature_map(n, reps=1, entanglement='linear')
    elif fmap_name == 'Pauli':
        fmap = pauli_feature_map(n, reps=1, paulis=['Z','XX'], entanglement='linear')
    else:
        raise ValueError(fmap_name)
    if ansatz_name == 'RealAmplitudes':
        ansatz = real_amplitudes(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'EfficientSU2':
        ansatz = efficient_su2(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'TwoLocal':
        ansatz = n_local(n, rotation_blocks=['ry','rz'], entanglement_blocks='cz', reps=reps, entanglement='linear')
    elif ansatz_name == 'PauliTwoDesign':
        ansatz = pauli_two_design(n, reps=reps, seed=33)
    else:
        raise ValueError(ansatz_name)
    return fmap, ansatz

def fit_model(kind, train_x, train_y, fmap_name='Z', ansatz_name='RealAmplitudes', maxiter=40, reps=1):
    fmap, ansatz = circuits(train_x.shape[1], fmap_name, ansatz_name, reps)
    initial = np.linspace(0.1,0.7,ansatz.num_parameters)
    history = []
    def optimizer(fun, x0, jac=None, bounds=None):
        def tracked(weights):
            value = float(fun(weights))
            history.append(value)
            return value
        return minimize(tracked, x0, method='COBYLA', options={'maxiter':maxiter, 'rhobeg':0.3})
    started = time.perf_counter()
    if kind == 'VQC':
        model = VQC(feature_map=fmap, ansatz=ansatz, sampler=QMLSampler(shots=None),
                    interpret=parity, output_shape=2, initial_point=initial, optimizer=optimizer)
        model.fit(train_x, train_y)
        def predict_probability(x):
            return np.asarray(model.neural_network.forward(x, model.weights))[:,1]
        weights = model.weights
    else:
        circuit = fmap.compose(ansatz)
        if kind == 'SQNN':
            qnn = SamplerQNN(circuit=circuit, sampler=QMLSampler(shots=None),
                             input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                             interpret=parity, output_shape=2)
            def probability(x,w):
                return np.asarray(qnn.forward(x,w))[:,1]
            start = initial
        elif kind == 'EQNN':
            qnn = EstimatorQNN(circuit=circuit, estimator=QMLEstimator(default_precision=0),
                              observables=SparsePauliOp('Z'*train_x.shape[1]),
                              input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                              default_precision=0)
            def probability(x,w):
                expectation = np.asarray(qnn.forward(x,w[:-2])).ravel()
                # 마지막 두 값은 고전 출력층의 가중치와 편향입니다.
                score = np.clip(w[-2]*expectation+w[-1],-40,40)
                return 1/(1+np.exp(-score))
            start = np.r_[initial,-1.0,0.0]
        else:
            raise ValueError(kind)
        def loss(w):
            p1 = np.clip(probability(train_x,w),1e-10,1-1e-10)
            value = float(-np.mean(train_y*np.log2(p1)+(1-train_y)*np.log2(1-p1)))
            return value
        result = optimizer(loss,start)
        weights = result.x
        def predict_probability(x):
            return probability(x,weights)
        model = None
    print(kind, fmap_name, ansatz_name, '학습 평가 수:',len(history),
          '최초/마지막 기록 손실:',round(history[0],6),round(history[-1],6))
    return predict_probability, model, {'parameters':len(weights),'depth':fmap.compose(ansatz).depth(),
                                       'seconds':time.perf_counter()-started,'history':history}

def report(name,y,p1):
    p1 = np.asarray(p1).ravel()
    assert p1.shape == y.shape and np.isfinite(p1).all()
    assert np.all((p1>=-1e-8)&(p1<=1+1e-8))
    predicted = (p1>=0.5).astype(int)
    result = {'model':name, 'accuracy':accuracy_score(y,predicted),
              'precision':precision_score(y,predicted,zero_division=0),
              'recall':recall_score(y,predicted,zero_division=0),
              'f1':f1_score(y,predicted,zero_division=0)}
    print({k:round(float(v),4) if k!='model' else v for k,v in result.items()})
    print('혼동행렬 [[TN,FP],[FN,TP]]:',confusion_matrix(y,predicted,labels=[0,1]).tolist())
    return result

def split_fixed(X,y):
    # 각 클래스에서 원래 행 순서의 네 번째마다 시험 표본으로 사용합니다.
    test = np.concatenate([np.flatnonzero(y==c)[3::4] for c in [0,1]])
    train = np.setdiff1d(np.arange(len(y)),test)
    assert not np.intersect1d(train,test).size
    assert len(np.unique(y[train]))==len(np.unique(y[test]))==2
    return train,test

print('버전:',{k:metadata.version(k) for k in ['qiskit','qiskit-machine-learning','numpy','scipy','scikit-learn']})

X = np.array([[0.2, 0.2, 0.8, 0.8],
 [0.8, 0.8, 0.2, 0.2],
 [0.4, 0.3, 0.7, 0.6],
 [0.6, 0.7, 0.3, 0.4],
 [0.7, 0.1, 0.9, 0.3],
 [0.3, 0.9, 0.1, 0.7],
 [0.5, 0.4, 0.8, 0.5],
 [0.5, 0.8, 0.4, 0.5],
 [0.1, 0.2, 0.6, 0.9],
 [0.9, 0.6, 0.2, 0.1],
 [0.8, 0.3, 0.6, 0.2],
 [0.2, 0.6, 0.3, 0.8],
 [0.3, 0.4, 0.7, 0.7],
 [0.7, 0.7, 0.4, 0.3],
 [0.6, 0.2, 0.7, 0.4],
 [0.4, 0.7, 0.2, 0.6],
 [0.9, 0.1, 0.8, 0.1],
 [0.1, 0.8, 0.1, 0.9],
 [0.4, 0.3, 0.5, 0.6],
 [0.6, 0.5, 0.3, 0.4],
 [0.2, 0.4, 0.6, 0.8],
 [0.8, 0.6, 0.4, 0.2],
 [0.7, 0.2, 0.5, 0.3],
 [0.3, 0.5, 0.2, 0.7]], dtype=float)
y = (X[:,1]+(1-X[:,2])>1).astype(int)
train,test = split_fixed(X,y)
scaler = MinMaxScaler(feature_range=(0,1),clip=True).fit(X[train])
train_x,test_x = scaler.transform(X[train]),scaler.transform(X[test])
print('학습/시험 크기:',train_x.shape,test_x.shape)
print('학습/시험 라벨 수:',np.bincount(y[train]).tolist(),np.bincount(y[test]).tolist())
import joblib
predict,model,info=fit_model('VQC',train_x,y[train],'Z','PauliTwoDesign',maxiter=80)
report('PreferenceVQC',y[test],predict(test_x))
model.to_dill('preference_model.dill')
joblib.dump(scaler,'preference_scaler.joblib')
restored=VQC.from_dill('preference_model.dill')
restored_scaler=joblib.load('preference_scaler.joblib')
restored_p=np.asarray(restored.neural_network.forward(test_x,restored.weights))[:,1]
assert np.allclose(predict(test_x),restored_p)
sample=np.array([[.6,.8,.3,.4]])
p1=float(restored.neural_network.forward(restored_scaler.transform(sample),restored.weights)[0,1])
print('저장/복원 확률 일치:',True)
print('단건 P(1)/그룹:',round(p1,6),'그룹 1' if p1>=.5 else '그룹 0')
print('시험 P(1):',np.round(restored_p,4).tolist())
import threading
from flask import Flask, request, render_template_string
from werkzeug.serving import make_server
from urllib.request import urlopen

app = Flask(__name__)
questions = ['사람들과 함께 있을 때 에너지가 납니다.',
             '새로운 가능성과 아이디어에 관심이 있습니다.',
             '판단할 때 논리적 기준을 중시합니다.',
             '계획을 세우고 일정에 맞추어 행동합니다.']
page = '''<!doctype html><html lang="ko"><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>양자 성향 점수 분류</title>
<style>body{font-family:sans-serif;background:#f0f4f8;margin:0;padding:32px}
main{max-width:680px;margin:auto;background:white;padding:28px;border-radius:16px}
label{display:block;margin:22px 0 8px}input{width:100%}
button{background:#2546a8;color:white;padding:12px 24px;border:0;border-radius:8px}
.result{padding:18px;background:#eef3ff;margin-top:20px}</style><main>
<h1>양자 성향 점수 분류</h1>
<p>응답 범위는 1~5입니다. 교육용 두 그룹 분류 결과를 확인합니다.</p>
<form method="post" action="/analyze">
{% for q in questions %}<label>{{loop.index}}. {{q}}</label>
<input type="range" name="q{{loop.index}}" min="1" max="5" step="1" value="3"
oninput="this.nextElementSibling.value=this.value"><output>3</output>{% endfor %}
<p><button type="submit">분류 결과 확인</button></p></form>
{% if result %}<section class="result"><h2>{{result.label}}</h2>
<p>그룹 1 출력 확률: {{result.probability}}</p><p>정규화 입력: {{result.inputs}}</p>
<p>그룹 기준은 직관 점수와 사고 점수의 관계입니다. 전체 MBTI 유형과 직업 적합성은 평가하지 않습니다.</p>
</section>{% endif %}</main></html>'''

@app.get('/')
def index():
    return render_template_string(page,questions=questions,result=None)

@app.post('/analyze')
def analyze():
    try:
        values=np.array([float(request.form[f'q{i}']) for i in range(1,5)])
        if not np.isfinite(values).all() or np.any((values<1)|(values>5)) or np.any(values!=np.floor(values)):
            raise ValueError('응답은 1~5 정수여야 합니다.')
    except (KeyError,ValueError):
        return '네 문항에 1~5 정수로 응답해 주세요.',400
    normalized=(values-1)/4
    transformed=restored_scaler.transform(normalized.reshape(1,-1))
    probability=float(restored.neural_network.forward(transformed,restored.weights)[0,1])
    result={'label':'그룹 1' if probability>=.5 else '그룹 0',
            'probability':f'{probability:.4f}', 'inputs':normalized.tolist()}
    return render_template_string(page,questions=questions,result=result)

client=app.test_client()
assert client.get('/').status_code==200
assert client.post('/analyze',data={'q1':'3','q2':'5','q3':'2','q4':'4'}).status_code==200
for invalid in ['0','6','nan','2.5']:
    assert client.post('/analyze',data={'q1':invalid,'q2':'3','q3':'3','q4':'3'}).status_code==400
assert client.post('/analyze',data={'q1':'3'}).status_code==400
print('웹 검사: 정상 요청 200, 범위·결측·비정수 입력 400')
if 'server' in globals():
    server.shutdown()
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate()
server=make_server('0.0.0.0',5000,app)
threading.Thread(target=server.serve_forever,daemon=True).start()
assert urlopen('http://127.0.0.1:5000/').status==200
print('내부 HTTP 검사: 200')

# Google Colab에서 공용 URL을 생성합니다.
import os, re, subprocess, platform
from urllib.request import urlretrieve
from pathlib import Path
binary=Path('/tmp/cloudflared_day33')
machine=platform.machine()
architecture='arm64' if machine=='aarch64' else 'amd64'
if not binary.exists():
    urlretrieve(f'https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-{architecture}',binary)
    binary.chmod(0o755)
log=Path('tunnel.log').open('w')
tunnel=subprocess.Popen([str(binary),'tunnel','--url','http://127.0.0.1:5000',
                         '--no-autoupdate','--protocol','http2'],stdout=log,stderr=log)
public_url=None
for _ in range(45):
    time.sleep(1)
    text=Path('tunnel.log').read_text()
    match=re.search(r'https://[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com',text)
    if match:
        public_url=match.group(0)
        break
    if tunnel.poll() is not None:
        break
if public_url:
    print('브라우저 접속 URL:',public_url)
else:
    print('공용 URL 생성 실패: tunnel.log에서 연결 상태를 확인합니다.')
    tunnel.terminate()
# 노트북의 Python 커널이 실행되는 동안 서버와 터널이 유지됩니다.

버전: {'qiskit': '2.2.3', 'qiskit-machine-learning': '0.9.0', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1'}
학습/시험 크기: (18, 4) (6, 4)
학습/시험 라벨 수: [9, 9] [3, 3]
VQC Z PauliTwoDesign 학습 평가 수: 80 최초/마지막 기록 손실: 0.995347 0.799833
{'model': 'PreferenceVQC', 'accuracy': 0.6667, 'precision': 1.0, 'recall': 0.3333, 'f1': 0.5}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [2, 1]]
저장/복원 확률 일치: True
단건 P(1)/그룹: 0.508153 그룹 1
시험 P(1): [0.3828, 0.4242, 0.4861, 0.5201, 0.4754, 0.3331]


INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:15:06] "GET / HTTP/1.1" 200 -


웹 검사: 정상 요청 200, 범위·결측·비정수 입력 400
내부 HTTP 검사: 200
브라우저 접속 URL: https://pending-acceptable-november-owners.trycloudflare.com


정상 입력 화면과 정상 분석 요청은 HTTP 200을 반환합니다. 범위 밖의 값, nan, 소수와 누락 입력은 HTTP 400을 반환합니다. 분류 화면의 확률은 학습된 모델의 출력이며 보정된 심리측정 확신도를 의미하지 않습니다.

공용 URL 생성 성공 시 출력된 HTTPS 주소에서 웹 화면을 엽니다. 생성에 실패하면 tunnel.log의 DNS·연결 오류를 확인합니다. 정상 동작 화면에는 네 응답값과 그룹·확률이 표시됩니다. 응답 화면과 결과 화면을 각각 캡처하여 입력값과 출력값을 확인합니다.